In [1]:
import sys

sys.path.append("..")

In [2]:
pwd

'c:\\Users\\user\\Documents\\AI-EngineeringRoadmap\\llm-from-scratch\\02_tiny_llm'

In [3]:
from model.causal_lm import GPTForCausalLM
from model.config import GPTConfig

config = GPTConfig(
    vocab_size=50257,
    context_length=1024,
    emb_dim=768,
    n_heads=12,
    n_layers=12,
    drop_rate=0.1,
    qkv_bias=True,
)

model = GPTForCausalLM(config)
model

GPTForCausalLM(
  (model): GPTModel(
    (tok_emb): Embedding(50257, 768)
    (pos_emb): Embedding(1024, 768)
    (drop_emb): Dropout(p=0.1, inplace=False)
    (trf_blocks): Sequential(
      (0): TransformerBlock(
        (attn): MultiHeadAttention(
          (W_query): Linear(in_features=768, out_features=768, bias=True)
          (W_key): Linear(in_features=768, out_features=768, bias=True)
          (W_value): Linear(in_features=768, out_features=768, bias=True)
          (out_proj): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (ffn): FeedForward(
          (net): Sequential(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Linear(in_features=3072, out_features=768, bias=True)
          )
        )
        (norm1): LayerNorm()
        (norm2): LayerNorm()
        (drop_shortcut): Dropout(p=0.1, inplace=False)
      )
      (1)

### Loading Pretrained Weights from OpenAI

In [4]:
from __future__ import annotations

import json
import logging
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
import requests
import tensorflow as tf
from tqdm import tqdm

In [6]:
VALID_MODEL_SIZES = ("124M", "355M", "774M", "1558M")
 
# Files that make up one GPT-2 checkpoint release.
CHECKPOINT_FILENAMES = (
    "checkpoint",
    "encoder.json",
    "hparams.json",
    "model.ckpt.data-00000-of-00001",
    "model.ckpt.index",
    "model.ckpt.meta",
    "vocab.bpe",
)

_BLOCK_PREFIX_RE = re.compile(r"^h(?P<layer>\d+)$")

In [ ]:
### Understanding Path concepts


[1]


In [22]:
p = Path("gpt2_checkpoints") / "124M" / "hparams.json"
print(p)

gpt2_checkpoints\124M\hparams.json


In [25]:
# check if a path exist
p = Path("some_file.txt")

In [26]:
p.exists()

False

In [29]:
pwd

'c:\\Users\\user\\Documents\\AI-EngineeringRoadmap\\llm-from-scratch\\02_tiny_llm'

In [30]:
import os

os.listdir(os.getcwd())

['data',
 'finetune_classification.py',
 'generate.py',
 'gpt2',
 'hyperparameter_tuning',
 'learning_rate_scheuler',
 'loading_pretrained_weight.ipynb',
 'load_pretrained_weight.py',
 'mlflow.db',
 'model',
 'README.md',
 'sampling.py',
 'tests',
 'tokenizers',
 'train.py',
 'user_interface',
 '__pycache__']

In [33]:
p = Path("gpt2/hparams.json")
info = p.stat()
print(f"The size of {p.name} is {info.st_size} bytes")  # size in bytes

The size of hparams.json is 90 bytes


In [34]:
info

os.stat_result(st_mode=33206, st_ino=4503599627629793, st_dev=1046530876, st_nlink=1, st_uid=0, st_gid=0, st_size=90, st_atime=1788350618, st_mtime=1788350618, st_ctime=1788350618)

In [35]:
## Getting the parent of a path
p = Path("gpt2_checkpoints/124M/hparams.json")
print(p.parent)
# gpt2_checkpoints/124M

gpt2_checkpoints\124M


In [ ]:
Path("a/b/c").mkdir(parents=True, exist_ok=True) # A way to create a path and also all that it should have

In [36]:
p = Path("output.bin")
### A file in a path can be opened and written to
with p.open("wb") as f:
    f.write(b"some bytes")

In [38]:
p = Path("gpt2_checkpoints/124M/hparams.json")

print(p.suffix)     # '.json'         — file extension
print(p.stem)       # 'hparams'       — filename without extension
print(p.is_file())   # True/False      — is this specifically a file (not a directory)?
print(p.is_dir())    # True/False      — is this specifically a directory?
print(str(p))        # convert back to a plain string, e.g. for libraries that don't accept Path objects

.json
hparams
False
False
gpt2_checkpoints\124M\hparams.json


In [11]:
@dataclass(frozen=True)
class MirrorSource:
    label: str
    base_url: str 

    def url_for(self, model_size: str, file_name: str) -> str:
        return f"{self.base_url}/{model_size}/{file_name}"

@dataclass(frozen=True)
class GPT2Source:
    mirrors: tuple[MirrorSource, ...] = field(
        default_factory=lambda: (
            MirrorSource("openai", "https://openaipublic.blob.core.windows.net/gpt-2/models"),
            MirrorSource("backup", "https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2"),
       )
    )

source = GPT2Source()
source.mirrors[0].base_url

'https://openaipublic.blob.core.windows.net/gpt-2/models'

In [15]:
for file_name in CHECKPOINT_FILENAMES:
    path = source.mirrors[0].url_for("124M", file_name)
    print(path)

https://openaipublic.blob.core.windows.net/gpt-2/models/124M/checkpoint
https://openaipublic.blob.core.windows.net/gpt-2/models/124M/encoder.json
https://openaipublic.blob.core.windows.net/gpt-2/models/124M/hparams.json
https://openaipublic.blob.core.windows.net/gpt-2/models/124M/model.ckpt.data-00000-of-00001
https://openaipublic.blob.core.windows.net/gpt-2/models/124M/model.ckpt.index
https://openaipublic.blob.core.windows.net/gpt-2/models/124M/model.ckpt.meta
https://openaipublic.blob.core.windows.net/gpt-2/models/124M/vocab.bpe


In [18]:
source = None or GPT2Source() # None is falsy, so Python returns the 2nd one
source.mirrors

(MirrorSource(label='openai', base_url='https://openaipublic.blob.core.windows.net/gpt-2/models'),
 MirrorSource(label='backup', base_url='https://f001.backblazeb2.com/file/LLMs-from-scratch/gpt2'))

In [19]:
class GPT2Downloader:
    def __init__(self, sources: GPT2Source | None = None, chunk_size_bytes: int = 1024):
        self.sources = sources or GPT2Source()
        self.chunk_size_bytes = chunk_size_bytes

    def fetch(self, model_size, file_name: str, destination: Path) -> None:
        last_error: Exception | None = None 

        for mirror in self.sources.mirrors:
            url = mirror.url_for(model_size, file_name)
            try:
                self._download_one(url, destination, mirror.label)
                return 
            except requests.exceptions.RequestException as exc:
                last_error = exc
                continue

        raise RuntimeError(
            f"Could not download '{file_name}' for model size '{model_size}' from any "
            f"configured mirror. Last error: {last_error}"
        )

    def _download_one(self, url, destination: Path, source_label: str) -> None:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()

        remote_size = int(response.headers.get("Content-Length", 0))
        if self._already_downloaded(destination, remote_size):
            print(f"Skipping {destination.name} already present, size matches")
            return

        destination.parent.mkdir(parents=True, exist_ok=True)
        with tqdm(total=remote_size, unit="iB", unit_scale=True, desc=f"[{source_label}] {destination.name}") as progress_bar:
            with destination.open("wb") as f:
                for chunk in response.iter_content(chunk_size=self.chunk_size_bytes):
                    if not chunk:
                        continue
                    f.write(chunk)
                    progress_bar.update(len(chunk))
            
    @staticmethod
    def _already_downloaded(destination:Path, remote_size: int) -> bool:
        if not destination.exists():
            return False
        return remote_size > 0 and destination.stat().st_size == remote_size

In [40]:
downloader = GPT2Downloader()
downloader.fetch("124M", "hparams.json", Path("gpt2/hparams.json"))
# Then check: did test_dl/hparams.json actually get created, and does its content look like real JSON?

Skipping hparams.json already present, size matches


In [21]:
Path("test_dl/hparams.json").name

'hparams.json'

In [46]:
class GPT2WeightLoader:
    def __init__(self, models_root: str | Path, downloader: GPT2Downloader | None = None):
        self.models_root = Path(models_root)
        self.downloader = downloader or GPT2Downloader()

    def load(self, model_size: str) -> tuple[dict[str, Any], dict[str, Any]]:
        if model_size not in VALID_MODEL_SIZES:
            raise ValueError(
                f"Unsupported model_size {model_size!r}; expected one of {VALID_MODEL_SIZES}"
            )

        model_dir = self.models_root / model_size
        self._ensure_downloaded(model_size, model_dir)
        settings_path = model_dir / "hparams.json"
        with settings_path.open("r", encoding="utf-8") as f:
            settings = json.load(f)

        checkpoint_path = tf.train.latest_checkpoint(str(model_dir))
        if checkpoint_path is None:
            raise FileNotFoundError(
                f"No TensorFlow checkpoint found in {model_dir} after download — "
                "the download may have failed partway through."
            )
 
        params = load_params_from_checkpoint(checkpoint_path, num_layers=settings["n_layer"])
        return settings, params

    def _ensure_downloaded(self, model_size: str, model_dir: Path) -> None:
        model_dir.mkdir(parents=True, exist_ok=True)
        for filename in CHECKPOINT_FILENAMES:
            self.downloader.fetch(model_size, filename, model_dir / filename)

def load_params_from_checkpoint(checkpoint_path, num_layers):
    ...

In [47]:
settings, params = GPT2WeightLoader("gpt2").load("124M")

[openai] checkpoint: 100%|██████████| 77.0/77.0 [00:00<00:00, 25.7kiB/s]
[openai] encoder.json: 100%|██████████| 1.04M/1.04M [00:05<00:00, 185kiB/s] 
[openai] hparams.json: 100%|██████████| 90.0/90.0 [00:00<00:00, 44.9kiB/s]
[openai] model.ckpt.data-00000-of-00001: 100%|██████████| 498M/498M [22:28<00:00, 369kiB/s]    
[openai] model.ckpt.index: 100%|██████████| 5.21k/5.21k [00:00<00:00, 1.74MiB/s]
[openai] model.ckpt.meta: 100%|██████████| 471k/471k [00:01<00:00, 386kiB/s]  
[openai] vocab.bpe: 100%|██████████| 456k/456k [00:01<00:00, 324kiB/s]  


In [48]:
settings

{'n_vocab': 50257, 'n_ctx': 1024, 'n_embd': 768, 'n_head': 12, 'n_layer': 12}

In [49]:
params

In [52]:
from model.config import GPTConfig
from dataclasses import asdict

GPT_CONFIG_124M = asdict(GPTConfig(50257, 256, 768, 12, 12))
GPT_CONFIG_124M

{'vocab_size': 50257,
 'context_length': 256,
 'emb_dim': 768,
 'n_heads': 12,
 'n_layers': 12,
 'drop_rate': 0.1,
 'qkv_bias': False}

In [53]:
from model.gpt_model import GPTModel

In [54]:
# Define model configurations in a dictionary for compactness
model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

# Copy the base configuration and update with specific model settings
model_name = "gpt2-small (124M)"  # Example model name
NEW_CONFIG = GPT_CONFIG_124M.copy()
NEW_CONFIG.update(model_configs[model_name])
NEW_CONFIG.update({"context_length": 1024, "qkv_bias": True})

gpt = GPTModel(NEW_CONFIG)
gpt.eval();

In [55]:
gpt

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(1024, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Linear(in_features=768, out_features=768, bias=True)
        (W_key): Linear(in_features=768, out_features=768, bias=True)
        (W_value): Linear(in_features=768, out_features=768, bias=True)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNorm()
      (norm2): LayerNorm()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (attn): MultiHeadAttention(
        (W_query): Li